# 07 — Trade Signal Rules v0.1

Translate the relative VRP tenor signal into baseline dynamic-tenor put/call trade signals.

This notebook does not price options, backtest P&L, optimize parameters, or calculate Kelly sizing. It only creates the trade-rule signal layer.


In [1]:
# ============================================================
# 07_trade_signal_rules_v0_1
#
# Goal:
#   Translate current 30d trading rules into dynamic-tenor
#   trade signal candidates across 9d–33d.
#
# Inputs:
#   vrp_zscore_panel_v0_1.parquet
#   preferred_vrp_tenor_by_date_v0_1.parquet
#
# Outputs:
#   trade_signal_panel_v0_1.parquet
#   daily_trade_signal_summary_v0_1.parquet
# ============================================================

from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

current_dir = Path.cwd()

if current_dir.name.lower() == "notebooks":
    PROJECT_ROOT = current_dir.parent
else:
    PROJECT_ROOT = current_dir

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DATA_DIR = DATA_DIR / "processed"
AUDIT_DIR = DATA_DIR / "audit"

TARGET_TENOR_DAYS = [9, 12, 15, 18, 21, 24, 27, 30, 33]

ZSCORE_PANEL_PARQUET = PROCESSED_DATA_DIR / "vrp_zscore_panel_v0_1.parquet"
PREFERRED_TENOR_PARQUET = PROCESSED_DATA_DIR / "preferred_vrp_tenor_by_date_v0_1.parquet"

TRADE_SIGNAL_PANEL_CSV = PROCESSED_DATA_DIR / "trade_signal_panel_v0_1.csv"
TRADE_SIGNAL_PANEL_PARQUET = PROCESSED_DATA_DIR / "trade_signal_panel_v0_1.parquet"

DAILY_SIGNAL_SUMMARY_CSV = PROCESSED_DATA_DIR / "daily_trade_signal_summary_v0_1.csv"
DAILY_SIGNAL_SUMMARY_PARQUET = PROCESSED_DATA_DIR / "daily_trade_signal_summary_v0_1.parquet"

SIGNAL_FREQUENCY_CSV = AUDIT_DIR / "trade_signal_frequency_v0_1.csv"
SIGNAL_STATE_SUMMARY_CSV = AUDIT_DIR / "signal_state_summary_v0_1.csv"

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Z-score panel exists:", ZSCORE_PANEL_PARQUET.exists())
print("Preferred tenor panel exists:", PREFERRED_TENOR_PARQUET.exists())

Project root: C:\Users\patri\vrp_project
Z-score panel exists: True
Preferred tenor panel exists: True


In [2]:
# ============================================================
# Load signal inputs
# ============================================================

zscore_df = pd.read_parquet(ZSCORE_PANEL_PARQUET).copy()
preferred_tenor_df = pd.read_parquet(PREFERRED_TENOR_PARQUET).copy()

required_zscore_cols = [
    "trade_date",
    "target_days",
    "vix_style_vol",
    "trailing_realized_vol",
    "trailing_realized_variance",
    "primary_vrp_signal",
    "vrp_trailing_variance_ratio",
    "vrp_trailing_vol_ratio",
    "vrp_log_ratio_3m_z",
    "vrp_log_ratio_1y_z",
    "vrp_log_ratio_blended_z",
    "spx_close",
    "spx_rsi_14",
]

missing_zscore_cols = [c for c in required_zscore_cols if c not in zscore_df.columns]

if missing_zscore_cols:
    raise ValueError(f"Missing columns in zscore_df: {missing_zscore_cols}")

required_preferred_cols = [
    "trade_date",
    "preferred_tenor",
    "preferred_source",
]

missing_preferred_cols = [c for c in required_preferred_cols if c not in preferred_tenor_df.columns]

if missing_preferred_cols:
    raise ValueError(f"Missing columns in preferred_tenor_df: {missing_preferred_cols}")

print("Z-score rows:", len(zscore_df))
print("Preferred tenor rows:", len(preferred_tenor_df))

print("\nZ-score date range:")
print(zscore_df["trade_date"].min(), "to", zscore_df["trade_date"].max())

print("\nPreferred tenor date range:")
print(preferred_tenor_df["trade_date"].min(), "to", preferred_tenor_df["trade_date"].max())

display(zscore_df.tail())
display(preferred_tenor_df.tail())

Z-score rows: 18099
Preferred tenor rows: 1969

Z-score date range:
20180625 to 20260625

Preferred tenor date range:
20180823 to 20260625


,trade_date,target_days,rate_symbol,rate_pct,implied_variance,vix_style_vol,near_root,near_expiration,near_days,near_variance,...,vrp_log_ratio_3m_std,vrp_log_ratio_3m_z,vrp_log_ratio_1y_mean,vrp_log_ratio_1y_std,vrp_log_ratio_1y_z,vrp_log_ratio_blended_z,rank_3m_z_highest_richest,rank_1y_z_highest_richest,rank_blended_z_highest_richest,rank_raw_log_ratio_highest
18094,20260625,21,SOFR,3.64,0.030683,17.516638,SPXW,20260710,15.000000,0.032300,...,0.445485,-1.505797,0.842629,0.524134,-1.955464,-1.730630,9.0,9.0,9.0,9.0
18095,20260625,24,SOFR,3.64,0.032504,18.028765,SPX,20260717,21.729167,0.030548,...,0.431185,-1.193720,0.849011,0.502968,-1.737542,-1.465631,8.0,8.0,8.0,8.0
18096,20260625,27,SOFR,3.64,0.034583,18.596577,SPX,20260717,21.729167,0.030548,...,0.411271,-0.925509,0.874895,0.475955,-1.519352,-1.222431,7.0,7.0,7.0,6.0
18097,20260625,30,SOFR,3.64,0.036027,18.980878,SPXW,20260724,29.000000,0.035731,...,0.397449,-0.577962,0.862363,0.447609,-1.298737,-0.938350,5.0,5.0,5.0,4.0
18098,20260625,33,SOFR,3.64,0.036810,19.185842,SPXW,20260724,29.000000,0.035731,...,0.396054,-0.418812,0.891583,0.437443,-1.166336,-0.792574,3.0,4.0,3.0,2.0


,trade_date,richest_blended_tenor,richest_blended_blended_z,richest_blended_variance_ratio,richest_blended_vol_ratio,richest_blended_spx_rsi_14,richest_interior_tenor,richest_interior_blended_z,richest_interior_variance_ratio,richest_interior_vol_ratio,richest_interior_spx_rsi_14,all_winner_is_endpoint,all_minus_interior_blended_z,use_endpoint_winner,preferred_tenor,preferred_source
1964,20260618,33,-1.437962,1.144294,1.069717,55.071587,30,-1.585211,1.035879,1.017781,55.071587,True,0.147249,False,30,interior_preferred
1965,20260622,9,-0.764554,1.123986,1.060182,53.235432,12,-1.203613,0.960250,0.979924,53.235432,True,0.439060,True,9,endpoint_allowed
1966,20260623,12,-0.426927,1.518207,1.232155,46.752076,12,-0.426927,1.518207,1.232155,46.752076,False,0.000000,False,12,interior_preferred
1967,20260624,9,-0.015593,1.763343,1.327909,46.342083,12,-0.416185,1.519085,1.232512,46.342083,True,0.400592,True,9,endpoint_allowed
1968,20260625,9,-0.140592,1.622403,1.273736,46.297996,12,-0.610471,1.341301,1.158146,46.297996,True,0.469879,True,9,endpoint_allowed


In [3]:
# ============================================================
# Baseline trade rule parameters
# ============================================================

# Core relative VRP threshold:
# primary_vrp_signal = log(implied variance / trailing realized variance)
PUT_MIN_PRIMARY_VRP_SIGNAL = 0.70

# Put tiers
PUT_TIER_A = {
    "name": "A_strongest",
    "min_realized_vol": 8.5,
    "min_blended_z": 0.70,
    "max_rsi": 70,
    "max_risk_pct": 3.9,
}

PUT_TIER_B = {
    "name": "B_good",
    "min_realized_vol": 6.8,
    "min_blended_z": 0.65,
    "max_rsi": 72,
    "max_risk_pct": 3.2,
}

PUT_TIER_C = {
    "name": "C_acceptable",
    "min_realized_vol": 6.8,
    "min_blended_z": 0.10,
    "max_rsi": 77,
    "max_risk_pct": 2.7,
}

# Call rules
CALL_MIN_BLENDED_Z = 0.10
CALL_MIN_RSI = 58

# Call sizing intentionally left unset for now.
# We'll define it later after call-spread backtest diagnostics.
CALL_MAX_RISK_PCT_PLACEHOLDER = np.nan

print("Put Tier A:", PUT_TIER_A)
print("Put Tier B:", PUT_TIER_B)
print("Put Tier C:", PUT_TIER_C)
print("Call minimum blended z:", CALL_MIN_BLENDED_Z)
print("Call minimum RSI:", CALL_MIN_RSI)

Put Tier A: {'name': 'A_strongest', 'min_realized_vol': 8.5, 'min_blended_z': 0.7, 'max_rsi': 70, 'max_risk_pct': 3.9}
Put Tier B: {'name': 'B_good', 'min_realized_vol': 6.8, 'min_blended_z': 0.65, 'max_rsi': 72, 'max_risk_pct': 3.2}
Put Tier C: {'name': 'C_acceptable', 'min_realized_vol': 6.8, 'min_blended_z': 0.1, 'max_rsi': 77, 'max_risk_pct': 2.7}
Call minimum blended z: 0.1
Call minimum RSI: 58


In [4]:
# ============================================================
# Build per-tenor trade signal panel
# ============================================================

signal_df = zscore_df.copy()

# Join preferred tenor by date so we can identify whether each tenor
# is the model's practical preferred tenor.
signal_df = signal_df.merge(
    preferred_tenor_df[
        [
            "trade_date",
            "preferred_tenor",
            "preferred_source",
        ]
    ],
    on="trade_date",
    how="left",
    validate="many_to_one",
)

signal_df["is_preferred_tenor"] = (
    signal_df["target_days"] == signal_df["preferred_tenor"]
)

# ----------------------------
# Put tier conditions
# ----------------------------

base_put_condition = (
    signal_df["primary_vrp_signal"] > PUT_MIN_PRIMARY_VRP_SIGNAL
)

put_tier_a_condition = (
    base_put_condition
    & (signal_df["trailing_realized_vol"] > PUT_TIER_A["min_realized_vol"])
    & (signal_df["vrp_log_ratio_blended_z"] > PUT_TIER_A["min_blended_z"])
    & (signal_df["spx_rsi_14"] < PUT_TIER_A["max_rsi"])
)

put_tier_b_condition = (
    base_put_condition
    & (signal_df["trailing_realized_vol"] > PUT_TIER_B["min_realized_vol"])
    & (signal_df["vrp_log_ratio_blended_z"] > PUT_TIER_B["min_blended_z"])
    & (signal_df["spx_rsi_14"] < PUT_TIER_B["max_rsi"])
)

put_tier_c_condition = (
    base_put_condition
    & (signal_df["trailing_realized_vol"] > PUT_TIER_C["min_realized_vol"])
    & (signal_df["vrp_log_ratio_blended_z"] > PUT_TIER_C["min_blended_z"])
    & (signal_df["spx_rsi_14"] < PUT_TIER_C["max_rsi"])
)

# Assign strongest qualifying tier first.
signal_df["put_tier"] = np.select(
    [
        put_tier_a_condition,
        put_tier_b_condition,
        put_tier_c_condition,
    ],
    [
        PUT_TIER_A["name"],
        PUT_TIER_B["name"],
        PUT_TIER_C["name"],
    ],
    default="none",
)

signal_df["put_tier_rank"] = np.select(
    [
        signal_df["put_tier"] == PUT_TIER_A["name"],
        signal_df["put_tier"] == PUT_TIER_B["name"],
        signal_df["put_tier"] == PUT_TIER_C["name"],
    ],
    [
        3,
        2,
        1,
    ],
    default=0,
)

signal_df["put_signal"] = signal_df["put_tier_rank"] > 0

signal_df["put_max_risk_pct"] = np.select(
    [
        signal_df["put_tier"] == PUT_TIER_A["name"],
        signal_df["put_tier"] == PUT_TIER_B["name"],
        signal_df["put_tier"] == PUT_TIER_C["name"],
    ],
    [
        PUT_TIER_A["max_risk_pct"],
        PUT_TIER_B["max_risk_pct"],
        PUT_TIER_C["max_risk_pct"],
    ],
    default=0.0,
)

# ----------------------------
# Call condition
# ----------------------------

signal_df["call_signal"] = (
    (signal_df["vrp_log_ratio_blended_z"] > CALL_MIN_BLENDED_Z)
    & (signal_df["spx_rsi_14"] > CALL_MIN_RSI)
)

signal_df["call_max_risk_pct"] = np.where(
    signal_df["call_signal"],
    CALL_MAX_RISK_PCT_PLACEHOLDER,
    0.0,
)

signal_df["call_short_strike_rule"] = np.where(
    signal_df["call_signal"],
    "+2sd_otm_short_call",
    "",
)

print("Rows:", len(signal_df))
print("Put signal rows:", signal_df["put_signal"].sum())
print("Call signal rows:", signal_df["call_signal"].sum())

display(
    signal_df[
        [
            "trade_date",
            "target_days",
            "preferred_tenor",
            "is_preferred_tenor",
            "primary_vrp_signal",
            "trailing_realized_vol",
            "vrp_log_ratio_blended_z",
            "spx_rsi_14",
            "put_signal",
            "put_tier",
            "put_max_risk_pct",
            "call_signal",
            "call_short_strike_rule",
        ]
    ].tail(45)
)

Rows: 18099
Put signal rows: 5207
Call signal rows: 4677


,trade_date,target_days,preferred_tenor,is_preferred_tenor,primary_vrp_signal,trailing_realized_vol,vrp_log_ratio_blended_z,spx_rsi_14,put_signal,put_tier,put_max_risk_pct,call_signal,call_short_strike_rule
18054,20260618,9,30.0,False,-0.845055,21.694354,-2.389979,55.071587,False,none,0.0,False,
18055,20260618,12,30.0,False,-0.497554,18.912899,-2.012917,55.071587,False,none,0.0,False,
18056,20260618,15,30.0,False,-0.710881,21.562294,-2.630850,55.071587,False,none,0.0,False,
18057,20260618,18,30.0,False,-0.517907,20.006952,-2.390546,55.071587,False,none,0.0,False,
18058,20260618,21,30.0,False,-0.336358,18.544916,-2.198142,55.071587,False,none,0.0,False,
18059,20260618,24,30.0,False,-0.195977,17.651477,-1.934820,55.071587,False,none,0.0,False,
18060,20260618,27,30.0,False,-0.038538,16.641972,-1.737345,55.071587,False,none,0.0,False,
18061,20260618,30,30.0,True,0.035251,16.288631,-1.585211,55.071587,False,none,0.0,False,
18062,20260618,33,30.0,False,0.134788,15.691578,-1.437962,55.071587,False,none,0.0,False,
18063,20260622,9,9.0,True,0.116881,15.347308,-0.764554,53.235432,False,none,0.0,False,


In [5]:
# ============================================================
# Signal frequency checks
# ============================================================

num_dates = signal_df["trade_date"].nunique()

print("Unique dates:", num_dates)

print("\nPut tier counts by row:")
display(signal_df["put_tier"].value_counts())

print("\nPut tier counts by tenor:")
display(
    signal_df
    .groupby(["target_days", "put_tier"])
    .size()
    .unstack(fill_value=0)
)

print("\nCall signal counts by tenor:")
display(
    signal_df
    .groupby("target_days")["call_signal"]
    .sum()
    .rename("call_signal_count")
    .to_frame()
)

# Dates with at least one put/call signal
daily_signal_presence = (
    signal_df
    .groupby("trade_date")
    .agg(
        any_put_signal=("put_signal", "any"),
        any_call_signal=("call_signal", "any"),
        num_put_tenors=("put_signal", "sum"),
        num_call_tenors=("call_signal", "sum"),
    )
    .reset_index()
)

print("\nDaily signal presence:")
display(daily_signal_presence.describe())

print("\nDates with at least one put signal:", daily_signal_presence["any_put_signal"].sum())
print("Dates with at least one call signal:", daily_signal_presence["any_call_signal"].sum())

display(daily_signal_presence.tail(30))

Unique dates: 2011

Put tier counts by row:


put_tier
none            12892
A_strongest      2198
C_acceptable     1814
B_good           1195
Name: count, dtype: int64


Put tier counts by tenor:


put_tier,A_strongest,B_good,C_acceptable,none
target_days,,,,
9,171,100,210,1530
12,193,122,211,1485
15,210,125,182,1494
18,224,134,221,1432
21,250,143,194,1424
24,264,137,203,1407
27,271,143,217,1380
30,298,145,190,1378
33,317,146,186,1362



Call signal counts by tenor:


,call_signal_count
target_days,
9,496
12,516
15,530
18,539
21,515
24,526
27,523
30,518
33,514



Daily signal presence:


,trade_date,num_put_tenors,num_call_tenors
count,2.011000e+03,2011.000000,2011.000000
mean,2.022040e+07,2.589259,2.325709
std,2.339183e+04,3.178185,3.492256
min,2.018062e+07,0.000000,0.000000
25%,2.020062e+07,0.000000,0.000000
50%,2.022062e+07,1.000000,0.000000
75%,2.024062e+07,5.000000,5.000000
max,2.026062e+07,9.000000,9.000000



Dates with at least one put signal: 1080
Dates with at least one call signal: 749


,trade_date,any_put_signal,any_call_signal,num_put_tenors,num_call_tenors
1981,20260513,True,True,6,6
1982,20260514,False,True,0,2
1983,20260515,True,True,1,1
1984,20260518,True,True,5,5
1985,20260519,True,True,6,6
1986,20260520,True,True,1,1
1987,20260521,False,False,0,0
1988,20260522,False,False,0,0
1989,20260526,True,True,4,4
1990,20260527,True,True,4,4


In [6]:
# ============================================================
# Select daily put and call candidates
# ============================================================

def select_best_put_candidate(group):
    eligible = group[group["put_signal"]].copy()

    if eligible.empty:
        return pd.Series({
            "selected_put_signal": False,
            "selected_put_tenor": np.nan,
            "selected_put_tier": "none",
            "selected_put_max_risk_pct": 0.0,
            "selected_put_primary_vrp_signal": np.nan,
            "selected_put_blended_z": np.nan,
            "selected_put_3m_z": np.nan,
            "selected_put_1y_z": np.nan,
            "selected_put_implied_vol": np.nan,
            "selected_put_trailing_rv": np.nan,
            "selected_put_variance_ratio": np.nan,
            "selected_put_vol_ratio": np.nan,
            "selected_put_is_preferred_tenor": False,
        })

    eligible = eligible.sort_values(
        [
            "put_tier_rank",
            "is_preferred_tenor",
            "vrp_log_ratio_blended_z",
            "primary_vrp_signal",
        ],
        ascending=[False, False, False, False],
    )

    row = eligible.iloc[0]

    return pd.Series({
        "selected_put_signal": True,
        "selected_put_tenor": row["target_days"],
        "selected_put_tier": row["put_tier"],
        "selected_put_max_risk_pct": row["put_max_risk_pct"],
        "selected_put_primary_vrp_signal": row["primary_vrp_signal"],
        "selected_put_blended_z": row["vrp_log_ratio_blended_z"],
        "selected_put_3m_z": row["vrp_log_ratio_3m_z"],
        "selected_put_1y_z": row["vrp_log_ratio_1y_z"],
        "selected_put_implied_vol": row["vix_style_vol"],
        "selected_put_trailing_rv": row["trailing_realized_vol"],
        "selected_put_variance_ratio": row["vrp_trailing_variance_ratio"],
        "selected_put_vol_ratio": row["vrp_trailing_vol_ratio"],
        "selected_put_is_preferred_tenor": row["is_preferred_tenor"],
    })


def select_best_call_candidate(group):
    eligible = group[group["call_signal"]].copy()

    if eligible.empty:
        return pd.Series({
            "selected_call_signal": False,
            "selected_call_tenor": np.nan,
            "selected_call_primary_vrp_signal": np.nan,
            "selected_call_blended_z": np.nan,
            "selected_call_3m_z": np.nan,
            "selected_call_1y_z": np.nan,
            "selected_call_implied_vol": np.nan,
            "selected_call_trailing_rv": np.nan,
            "selected_call_variance_ratio": np.nan,
            "selected_call_vol_ratio": np.nan,
            "selected_call_is_preferred_tenor": False,
            "selected_call_short_strike_rule": "",
        })

    eligible = eligible.sort_values(
        [
            "is_preferred_tenor",
            "vrp_log_ratio_blended_z",
            "primary_vrp_signal",
        ],
        ascending=[False, False, False],
    )

    row = eligible.iloc[0]

    return pd.Series({
        "selected_call_signal": True,
        "selected_call_tenor": row["target_days"],
        "selected_call_primary_vrp_signal": row["primary_vrp_signal"],
        "selected_call_blended_z": row["vrp_log_ratio_blended_z"],
        "selected_call_3m_z": row["vrp_log_ratio_3m_z"],
        "selected_call_1y_z": row["vrp_log_ratio_1y_z"],
        "selected_call_implied_vol": row["vix_style_vol"],
        "selected_call_trailing_rv": row["trailing_realized_vol"],
        "selected_call_variance_ratio": row["vrp_trailing_variance_ratio"],
        "selected_call_vol_ratio": row["vrp_trailing_vol_ratio"],
        "selected_call_is_preferred_tenor": row["is_preferred_tenor"],
        "selected_call_short_strike_rule": row["call_short_strike_rule"],
    })


daily_put_selection_df = (
    signal_df
    .groupby("trade_date")
    .apply(select_best_put_candidate)
    .reset_index()
)

daily_call_selection_df = (
    signal_df
    .groupby("trade_date")
    .apply(select_best_call_candidate)
    .reset_index()
)

daily_market_features_df = (
    signal_df
    .groupby("trade_date")
    .agg(
        spx_close=("spx_close", "first"),
        spx_rsi_14=("spx_rsi_14", "first"),
        preferred_tenor=("preferred_tenor", "first"),
        preferred_source=("preferred_source", "first"),
        num_put_tenors=("put_signal", "sum"),
        num_call_tenors=("call_signal", "sum"),
    )
    .reset_index()
)

daily_signal_summary_df = (
    daily_market_features_df
    .merge(daily_put_selection_df, on="trade_date", how="left")
    .merge(daily_call_selection_df, on="trade_date", how="left")
    .sort_values("trade_date")
    .reset_index(drop=True)
)

daily_signal_summary_df["both_put_and_call_signal"] = (
    daily_signal_summary_df["selected_put_signal"]
    & daily_signal_summary_df["selected_call_signal"]
)

daily_signal_summary_df["any_trade_signal"] = (
    daily_signal_summary_df["selected_put_signal"]
    | daily_signal_summary_df["selected_call_signal"]
)

print("Daily rows:", len(daily_signal_summary_df))
print("Selected put signal dates:", daily_signal_summary_df["selected_put_signal"].sum())
print("Selected call signal dates:", daily_signal_summary_df["selected_call_signal"].sum())
print("Both put and call signal dates:", daily_signal_summary_df["both_put_and_call_signal"].sum())
print("Any trade signal dates:", daily_signal_summary_df["any_trade_signal"].sum())

display(daily_signal_summary_df.tail(40))

C:\Users\patri\AppData\Local\Temp\ipykernel_24352\1194696391.py:103: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(select_best_put_candidate)


Daily rows: 2011
Selected put signal dates: 1080
Selected call signal dates: 749
Both put and call signal dates: 584
Any trade signal dates: 1245


C:\Users\patri\AppData\Local\Temp\ipykernel_24352\1194696391.py:110: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(select_best_call_candidate)


,trade_date,spx_close,spx_rsi_14,preferred_tenor,preferred_source,num_put_tenors,num_call_tenors,selected_put_signal,selected_put_tenor,selected_put_tier,...,selected_call_3m_z,selected_call_1y_z,selected_call_implied_vol,selected_call_trailing_rv,selected_call_variance_ratio,selected_call_vol_ratio,selected_call_is_preferred_tenor,selected_call_short_strike_rule,both_put_and_call_signal,any_trade_signal
1971,20260429,7135.95,66.585933,12.0,interior_preferred,4,4,True,12.0,A_strongest,...,0.884639,0.833139,17.422249,8.900762,3.831370,1.957389,True,+2sd_otm_short_call,True,True
1972,20260430,7209.01,70.210998,12.0,interior_preferred,0,0,False,NaN,none,...,NaN,NaN,NaN,NaN,NaN,NaN,False,,False,False
1973,20260501,7230.12,71.183782,9.0,endpoint_allowed,0,0,False,NaN,none,...,NaN,NaN,NaN,NaN,NaN,NaN,False,,False,False
1974,20260504,7200.75,67.863337,12.0,interior_preferred,6,6,True,12.0,B_good,...,0.879808,0.856274,16.720195,8.451871,3.913605,1.978283,True,+2sd_otm_short_call,True,True
1975,20260505,7259.22,70.785030,18.0,interior_preferred,5,5,True,18.0,C_acceptable,...,0.571118,0.493318,16.475916,9.602682,2.943839,1.715762,True,+2sd_otm_short_call,True,True
1976,20260506,7365.12,75.185395,18.0,interior_preferred,0,0,False,NaN,none,...,NaN,NaN,NaN,NaN,NaN,NaN,False,,False,False
1977,20260507,7337.11,72.092432,15.0,interior_preferred,0,0,False,NaN,none,...,NaN,NaN,NaN,NaN,NaN,NaN,False,,False,False
1978,20260508,7398.93,74.578135,30.0,interior_preferred,0,0,False,NaN,none,...,NaN,NaN,NaN,NaN,NaN,NaN,False,,False,False
1979,20260511,7412.84,75.115221,24.0,interior_preferred,7,7,True,24.0,C_acceptable,...,0.516379,0.370599,17.804969,10.700932,2.768466,1.663871,True,+2sd_otm_short_call,True,True
1980,20260512,7400.96,73.683428,24.0,interior_preferred,4,4,True,24.0,C_acceptable,...,0.394858,0.255037,17.317442,10.719197,2.610015,1.615554,True,+2sd_otm_short_call,True,True


In [7]:
# ============================================================
# Daily signal frequency diagnostics
# ============================================================

num_dates = len(daily_signal_summary_df)

signal_frequency_summary = pd.DataFrame({
    "metric": [
        "selected_put_signal",
        "selected_call_signal",
        "both_put_and_call_signal",
        "any_trade_signal",
    ],
    "count": [
        daily_signal_summary_df["selected_put_signal"].sum(),
        daily_signal_summary_df["selected_call_signal"].sum(),
        daily_signal_summary_df["both_put_and_call_signal"].sum(),
        daily_signal_summary_df["any_trade_signal"].sum(),
    ],
})

signal_frequency_summary["pct_of_dates"] = (
    signal_frequency_summary["count"] / num_dates
)

display(signal_frequency_summary)

print("Selected put tenor frequency:")
display(
    daily_signal_summary_df
    .dropna(subset=["selected_put_tenor"])
    ["selected_put_tenor"]
    .value_counts()
    .sort_index()
    .rename("count")
    .to_frame()
)

print("Selected put tier frequency:")
display(
    daily_signal_summary_df["selected_put_tier"]
    .value_counts()
    .rename("count")
    .to_frame()
)

print("Selected call tenor frequency:")
display(
    daily_signal_summary_df
    .dropna(subset=["selected_call_tenor"])
    ["selected_call_tenor"]
    .value_counts()
    .sort_index()
    .rename("count")
    .to_frame()
)

,metric,count,pct_of_dates
0,selected_put_signal,1080,0.537046
1,selected_call_signal,749,0.372452
2,both_put_and_call_signal,584,0.290403
3,any_trade_signal,1245,0.619095


Selected put tenor frequency:


,count
selected_put_tenor,
9.0,87
12.0,114
15.0,91
18.0,88
21.0,89
24.0,93
27.0,109
30.0,263
33.0,146


Selected put tier frequency:


,count
selected_put_tier,
none,931
A_strongest,564
C_acceptable,277
B_good,239


Selected call tenor frequency:


,count
selected_call_tenor,
9.0,70
12.0,115
15.0,82
18.0,75
21.0,74
24.0,61
27.0,85
30.0,155
33.0,32


In [8]:
# ============================================================
# Put/call overlap diagnostics
# ============================================================

overlap_df = daily_signal_summary_df[
    daily_signal_summary_df["both_put_and_call_signal"]
].copy()

print("Overlap dates:", len(overlap_df))
print("Overlap pct of all dates:", len(overlap_df) / len(daily_signal_summary_df))

print("\nOverlap by selected put tier:")
display(
    overlap_df["selected_put_tier"]
    .value_counts()
    .rename("count")
    .to_frame()
)

print("\nOverlap RSI summary:")
display(overlap_df["spx_rsi_14"].describe())

print("\nLatest overlap dates:")
display(
    overlap_df[
        [
            "trade_date",
            "spx_close",
            "spx_rsi_14",
            "selected_put_tenor",
            "selected_put_tier",
            "selected_put_max_risk_pct",
            "selected_put_blended_z",
            "selected_call_tenor",
            "selected_call_blended_z",
            "preferred_tenor",
        ]
    ].tail(30)
)

Overlap dates: 584
Overlap pct of all dates: 0.2904027846842367

Overlap by selected put tier:


,count
selected_put_tier,
A_strongest,240
B_good,189
C_acceptable,155



Overlap RSI summary:


count    584.000000
mean      65.525599
std        4.730512
min       58.018938
25%       61.439921
50%       65.277568
75%       68.786258
max       76.926862
Name: spx_rsi_14, dtype: float64


Latest overlap dates:


,trade_date,spx_close,spx_rsi_14,selected_put_tenor,selected_put_tier,selected_put_max_risk_pct,selected_put_blended_z,selected_call_tenor,selected_call_blended_z,preferred_tenor
1885,20251223,6909.79,59.693073,27.0,C_acceptable,2.7,0.202628,27.0,0.202628,27.0
1887,20251226,6929.94,61.078683,30.0,C_acceptable,2.7,0.391772,30.0,0.391772,30.0
1888,20251229,6905.74,58.111360,33.0,B_good,3.2,0.735154,9.0,0.904110,9.0
1893,20260106,6944.82,60.547422,33.0,A_strongest,3.9,0.737190,18.0,1.191353,18.0
1896,20260109,6966.28,61.735181,21.0,B_good,3.2,1.341191,21.0,1.341191,21.0
1897,20260112,6977.27,62.705228,33.0,A_strongest,3.9,1.019672,24.0,2.142502,24.0
1898,20260113,6963.74,60.666193,33.0,A_strongest,3.9,1.262822,24.0,2.310484,24.0
1907,20260127,6978.60,58.561546,33.0,B_good,3.2,0.657915,30.0,0.424751,30.0
1908,20260128,6978.03,58.489398,33.0,A_strongest,3.9,0.714669,30.0,0.533218,30.0
1965,20260421,7064.01,67.298565,12.0,C_acceptable,2.7,0.291309,12.0,0.291309,12.0


In [9]:
# ============================================================
# Daily signal state classification
# ============================================================

daily_signal_summary_df["signal_state"] = np.select(
    [
        daily_signal_summary_df["selected_put_signal"]
        & daily_signal_summary_df["selected_call_signal"],

        daily_signal_summary_df["selected_put_signal"]
        & ~daily_signal_summary_df["selected_call_signal"],

        ~daily_signal_summary_df["selected_put_signal"]
        & daily_signal_summary_df["selected_call_signal"],
    ],
    [
        "both_put_and_call",
        "put_only",
        "call_only",
    ],
    default="none",
)

signal_state_summary = (
    daily_signal_summary_df["signal_state"]
    .value_counts()
    .rename("count")
    .to_frame()
)

signal_state_summary["pct_of_dates"] = (
    signal_state_summary["count"] / len(daily_signal_summary_df)
)

display(signal_state_summary)

display(
    daily_signal_summary_df[
        [
            "trade_date",
            "spx_close",
            "spx_rsi_14",
            "signal_state",
            "selected_put_signal",
            "selected_put_tenor",
            "selected_put_tier",
            "selected_put_max_risk_pct",
            "selected_put_blended_z",
            "selected_call_signal",
            "selected_call_tenor",
            "selected_call_blended_z",
            "preferred_tenor",
        ]
    ].tail(60)
)

,count,pct_of_dates
signal_state,,
none,766,0.380905
both_put_and_call,584,0.290403
put_only,496,0.246643
call_only,165,0.082049


,trade_date,spx_close,spx_rsi_14,signal_state,selected_put_signal,selected_put_tenor,selected_put_tier,selected_put_max_risk_pct,selected_put_blended_z,selected_call_signal,selected_call_tenor,selected_call_blended_z,preferred_tenor
1951,20260331,6528.52,42.633907,none,False,NaN,none,0.0,NaN,False,NaN,NaN,24.0
1952,20260401,6575.32,45.690525,none,False,NaN,none,0.0,NaN,False,NaN,NaN,24.0
1953,20260402,6582.69,46.176889,none,False,NaN,none,0.0,NaN,False,NaN,NaN,27.0
1954,20260406,6611.83,48.153896,none,False,NaN,none,0.0,NaN,False,NaN,NaN,24.0
1955,20260407,6616.85,48.504814,none,False,NaN,none,0.0,NaN,False,NaN,NaN,12.0
1956,20260408,6782.81,58.504275,none,False,NaN,none,0.0,NaN,False,NaN,NaN,12.0
1957,20260409,6824.66,60.582879,none,False,NaN,none,0.0,NaN,False,NaN,NaN,9.0
1958,20260410,6816.89,59.982120,none,False,NaN,none,0.0,NaN,False,NaN,NaN,9.0
1959,20260413,6886.24,63.464495,none,False,NaN,none,0.0,NaN,False,NaN,NaN,12.0
1960,20260414,6967.38,67.074645,none,False,NaN,none,0.0,NaN,False,NaN,NaN,12.0


In [10]:
# ============================================================
# Provisional daily risk fields
# ============================================================

# Put-side risk is defined from your existing tiers.
daily_signal_summary_df["provisional_put_risk_pct"] = np.where(
    daily_signal_summary_df["selected_put_signal"],
    daily_signal_summary_df["selected_put_max_risk_pct"],
    0.0,
)

# Call-side risk is deliberately unset for now.
# We do not yet have a calibrated call max-risk framework.
daily_signal_summary_df["provisional_call_risk_pct"] = np.where(
    daily_signal_summary_df["selected_call_signal"],
    np.nan,
    0.0,
)

# Combined risk is not finalized when both sides fire.
# For now, use put risk as the only calibrated component.
daily_signal_summary_df["provisional_calibrated_risk_pct"] = (
    daily_signal_summary_df["provisional_put_risk_pct"]
)

daily_signal_summary_df["needs_call_risk_calibration"] = (
    daily_signal_summary_df["selected_call_signal"]
)

display(
    daily_signal_summary_df[
        [
            "trade_date",
            "signal_state",
            "selected_put_tier",
            "selected_put_tenor",
            "provisional_put_risk_pct",
            "selected_call_tenor",
            "provisional_call_risk_pct",
            "provisional_calibrated_risk_pct",
            "needs_call_risk_calibration",
            "spx_rsi_14",
        ]
    ].tail(60)
)

,trade_date,signal_state,selected_put_tier,selected_put_tenor,provisional_put_risk_pct,selected_call_tenor,provisional_call_risk_pct,provisional_calibrated_risk_pct,needs_call_risk_calibration,spx_rsi_14
1951,20260331,none,none,NaN,0.0,NaN,0.0,0.0,False,42.633907
1952,20260401,none,none,NaN,0.0,NaN,0.0,0.0,False,45.690525
1953,20260402,none,none,NaN,0.0,NaN,0.0,0.0,False,46.176889
1954,20260406,none,none,NaN,0.0,NaN,0.0,0.0,False,48.153896
1955,20260407,none,none,NaN,0.0,NaN,0.0,0.0,False,48.504814
1956,20260408,none,none,NaN,0.0,NaN,0.0,0.0,False,58.504275
1957,20260409,none,none,NaN,0.0,NaN,0.0,0.0,False,60.582879
1958,20260410,none,none,NaN,0.0,NaN,0.0,0.0,False,59.982120
1959,20260413,none,none,NaN,0.0,NaN,0.0,0.0,False,63.464495
1960,20260414,none,none,NaN,0.0,NaN,0.0,0.0,False,67.074645


In [11]:
# ============================================================
# Latest daily signal snapshot
# ============================================================

latest_trade_date = daily_signal_summary_df["trade_date"].max()

latest_daily_signal_df = daily_signal_summary_df[
    daily_signal_summary_df["trade_date"] == latest_trade_date
].copy()

print("Latest trade date:", latest_trade_date)

display(
    latest_daily_signal_df[
        [
            "trade_date",
            "spx_close",
            "spx_rsi_14",
            "preferred_tenor",
            "preferred_source",
            "signal_state",
            "selected_put_signal",
            "selected_put_tenor",
            "selected_put_tier",
            "selected_put_max_risk_pct",
            "selected_put_primary_vrp_signal",
            "selected_put_blended_z",
            "selected_call_signal",
            "selected_call_tenor",
            "selected_call_blended_z",
            "both_put_and_call_signal",
            "any_trade_signal",
            "provisional_put_risk_pct",
            "provisional_call_risk_pct",
            "provisional_calibrated_risk_pct",
            "needs_call_risk_calibration",
        ]
    ]
)


Latest trade date: 20260625


,trade_date,spx_close,spx_rsi_14,preferred_tenor,preferred_source,signal_state,selected_put_signal,selected_put_tenor,selected_put_tier,selected_put_max_risk_pct,...,selected_put_blended_z,selected_call_signal,selected_call_tenor,selected_call_blended_z,both_put_and_call_signal,any_trade_signal,provisional_put_risk_pct,provisional_call_risk_pct,provisional_calibrated_risk_pct,needs_call_risk_calibration
2010,20260625,7357.49,46.297996,9.0,endpoint_allowed,none,False,NaN,none,0.0,...,NaN,False,NaN,NaN,False,False,0.0,0.0,0.0,False


In [12]:
# ============================================================
# Final save after adding signal_state and provisional risk fields
# ============================================================

SIGNAL_FREQUENCY_CSV = AUDIT_DIR / "trade_signal_frequency_v0_1.csv"
SIGNAL_STATE_SUMMARY_CSV = AUDIT_DIR / "signal_state_summary_v0_1.csv"

signal_df.to_csv(TRADE_SIGNAL_PANEL_CSV, index=False)
signal_df.to_parquet(TRADE_SIGNAL_PANEL_PARQUET, index=False)

daily_signal_summary_df.to_csv(DAILY_SIGNAL_SUMMARY_CSV, index=False)
daily_signal_summary_df.to_parquet(DAILY_SIGNAL_SUMMARY_PARQUET, index=False)

signal_frequency_summary.to_csv(SIGNAL_FREQUENCY_CSV, index=False)
signal_state_summary.to_csv(SIGNAL_STATE_SUMMARY_CSV, index=True)

print("Saved trade signal panel CSV:", TRADE_SIGNAL_PANEL_CSV)
print("Saved trade signal panel parquet:", TRADE_SIGNAL_PANEL_PARQUET)

print("Saved daily signal summary CSV:", DAILY_SIGNAL_SUMMARY_CSV)
print("Saved daily signal summary parquet:", DAILY_SIGNAL_SUMMARY_PARQUET)

print("Saved signal frequency audit:", SIGNAL_FREQUENCY_CSV)
print("Saved signal state summary:", SIGNAL_STATE_SUMMARY_CSV)


Saved trade signal panel CSV: C:\Users\patri\vrp_project\data\processed\trade_signal_panel_v0_1.csv
Saved trade signal panel parquet: C:\Users\patri\vrp_project\data\processed\trade_signal_panel_v0_1.parquet
Saved daily signal summary CSV: C:\Users\patri\vrp_project\data\processed\daily_trade_signal_summary_v0_1.csv
Saved daily signal summary parquet: C:\Users\patri\vrp_project\data\processed\daily_trade_signal_summary_v0_1.parquet
Saved signal frequency audit: C:\Users\patri\vrp_project\data\audit\trade_signal_frequency_v0_1.csv
Saved signal state summary: C:\Users\patri\vrp_project\data\audit\signal_state_summary_v0_1.csv
